# Sudoku Knowledge Representation & Inference

You will implement three functions -- `build_general_kb`, `build_definite_kb`, `pl_bc_entails` -- and the full-grid solving logic in `sudoku_solver.py`.

This notebook imports and tests those functions. The Streamlit app (`sudoku_app.py`) must import the same implementation from `sudoku_solver.py`; do not copy or rewrite the solver functions inside the app.

**Rules:**

- In `sudoku_solver.py`, import only from `utils.py` and `logic_.py`; do not modify either file.
- Do not duplicate the core solver functions in this notebook or `sudoku_app.py`.
- All other content in this notebook may be edited freely.


In [1]:
from utils import *
from logic_ import *
import json
import time
import importlib
import sudoku_solver

# Reload so edits made to sudoku_solver.py are picked up when this cell is rerun.
importlib.reload(sudoku_solver)

from sudoku_solver import (
    atom,
    build_general_kb,
    build_definite_kb,
    solve_full_grid_fc,
    pl_bc_entails,
    solve_full_grid_bc,
)


## Loading a puzzle from JSON

Puzzles are provided as JSON, not embedded in this notebook. Each file looks like:

```json
{
  "n": 9, "box_h": 3, "box_w": 3,
  "puzzles": [
    {
      "givens": {"1_1": 3, "2_3": 1, ...},
      "given_count": 28,
      "solution": {"1_1": 3, "1_2": 4, ...}
    },
    ...
  ]
}
```

`"r_c"` string keys map to the value at row `r`, column `c` (1-indexed). `given_count` is exactly how many cells are given. `solution` is included so you can check your own work as you go, but your functions must not read `solution` to answer a query, they should only read `givens`.

In [2]:
#do not change this function, it is used to load the puzzle pool from a json file
def load_pool(path):
    with open(path) as f:
        raw = json.load(f)
    puzzles = []
    for p in raw['puzzles']:
        givens = {tuple(int(x) for x in k.split('_')): v for k, v in p['givens'].items()}
        solution = {tuple(int(x) for x in k.split('_')): v for k, v in p['solution'].items()}
        puzzles.append({'givens': givens, 'solution': solution, 'given_count': p['given_count']})
    return raw['n'], raw['box_h'], raw['box_w'], puzzles

n, box_h, box_w, puzzle_pool = load_pool('puzzles.json')
print(f'{len(puzzle_pool)} puzzles loaded, {n}x{n} grid, {box_h}x{box_w} boxes')
print('given_count values:', sorted(p['given_count'] for p in puzzle_pool))

# Pick one puzzle to work with through the rest of this notebook.
puzzle = puzzle_pool[0]
givens = puzzle['givens']
print(f"working puzzle has {puzzle['given_count']} givens")
for r in range(1, n + 1):
    print([givens.get((r, c), '.') for c in range(1, n + 1)])

5 puzzles loaded, 9x9 grid, 3x3 boxes
given_count values: [30, 33, 36, 39, 42]
working puzzle has 30 givens
['.', 3, '.', '.', '.', '.', '.', '.', '.']
[2, '.', '.', '.', '.', 7, 8, 6, '.']
[5, 8, '.', 2, 6, '.', '.', 3, '.']
[7, 5, '.', '.', '.', '.', '.', 8, '.']
['.', '.', '.', '.', 7, '.', 5, '.', 4]
['.', '.', '.', 5, 3, '.', '.', 9, 6]
['.', 1, 2, '.', '.', 9, '.', '.', '.']
[6, 4, '.', '.', 5, 8, 9, '.', '.']
['.', '.', '.', '.', 2, 3, '.', '.', '.']


Notice: `pl_fc_entails`/`pl_resolution`/`tt_entails` are all given to you, already implemented, in `logic_.py`. The one algorithm you write yourself in this assignment is **backward chaining** (`pl_bc_entails`) -- see Task 2.

## Define Symbols

Two families of propositional symbols, for row $r$, column $c$, value $v$ (all ranging over $1$ to $n$):

| Symbol | Meaning |
|---|---|
| $\mathit{Is}_{rcv}$ | cell $(r,c)$ has value $v$ |
| $\mathit{Not}_{rcv}$ | cell $(r,c)$ does **not** have value $v$ |

For each representation in Task 1, determine which of these two families is actually needed.

Hint: consider what form a definite (Horn) clause must take, and what `PropDefiniteKB.tell()` will accept.

### Helper function

`atom(prefix, r, c, v)`, where prefix can be either `Is` or `Not`, is a naming helper so propositional symbols need not be typed. It is provided for you in `sudoku_solver.py`; do not change it.

In code, $\mathit{Is}_{rcv}$ and $\mathit{Not}_{rcv}$ are written as single-word symbol names, e.g. `Is3_2_4` and `Not3_2_4`: the prefix is followed immediately by `r`, then by `c` and `v` separated by underscores. No separator is needed between the prefix and `r` since `expr()` only requires a symbol name to start with an uppercase letter. So `Is3_2_4` is parsed as one valid symbol.


In [3]:
# atom() is provided in sudoku_solver.py and imported above.


## Part A: Design the Sudoku Solver

### A.1) Knowledge Representation - Build KB

Every well-posed Sudoku puzzle satisfies exactly these conditions:

- Each cell is assigned **at least one** value from $\{1, \dots, n\}$.
- Each cell is assigned **at most one** value from $\{1, \dots, n\}$, i.e., it cannot hold two different values at once.
- No two cells in the same row hold the same value.
- No two cells in the same column hold the same value.
- No two cells in the same box hold the same value.
- The **givens** cells hold their stated values.

Formalize these Sudoku constraints as propositional logic, in **two** representations:

**(a) General clauses -  `build_general_kb`.**

Encode each of the following directly: no restriction here; you may use arbitrary disjunctions of positive or negated literals. Must return a `PropKB`. 

**(b) Definite (Horn) clauses:- `build_definite_kb`.** Must return a `PropDefiniteKB`. Recall a definite clause is a disjunction of literals with exactly one *positive* literal.

Equivalently written as an implication whose conclusion is a single positive literal and whose premises are a conjunction of positive literals: `P1 & P2 & ... & Pk ==> Q`.

`PropDefiniteKB.tell()` will reject anything else.

Both functions take the puzzle's givens as fixed facts.

- **Implement `build_general_kb()` and `build_definite_kb()` in sudoku_solver.py**
- **Rerun the import cell near the top of this notebook after making changes.**
- **Explain your representation in Conceptual Question 1 below.**

In [4]:
general_kb = build_general_kb(n, box_h, box_w, givens)
definite_kb = build_definite_kb(n, box_h, box_w, givens)

print(f"General KB clauses: {len(general_kb.clauses):,}")
print(f"Definite KB clauses/facts: {len(definite_kb.clauses):,}")


General KB clauses: 10,317
Definite KB clauses/facts: 21,171


### A.2) Solve the Puzzle

**(a) Resolution and model checking on the general representation.** Using `build_general_kb` and the library's `pl_resolution` / `tt_entails`, try to solve the puzzle, i.e., for each cell, determine which value is entailed. Attempt this in the code cell below: it is provided commented out, because both are sound and complete on `build_general_kb`'s output but neither scales to the full grid, and it is expected to hang or take an impractically long time. Uncomment a few lines at a time and give each at most about 30 seconds; use Kernel > Interrupt if it hasn't returned by then, and note what you observed.

Explain in your own words: what specifically makes `pl_resolution`'s cost grow out of control here, and separately, what makes `tt_entails`'s cost grow out of control? A simple complexity argument for each is the expected answer.

**Use your observations in Conceptual Question 2 below.**


In [5]:
# Attempt (a): try solving the puzzle using pl_resolution / tt_entails on the
# general KB. Left commented out because it is expected to take an
# impractically long time (pl_resolution) or be outright infeasible
# (tt_entails) on the full grid.
#
# Uncomment a few lines at a time and give each at most ~30 seconds; use
# Kernel > Interrupt if it hasn't returned by then.
#
# general_kb = build_general_kb(n, box_h, box_w, givens)
# r, c, v = 1, 1, givens.get((1, 1), 1)  # try one cell/value pair
# query = atom('Is', r, c, v)
#
# print(pl_resolution(general_kb, query))    
# print(tt_entails(associate('&', general_kb.clauses), query)) 

### Observation notes

Record what you observed from the resolution/model-checking experiment here. Use these observations when answering Conceptual Question 2 below.


The selected puzzle's general representation contains 10,317 CNF clauses over 729 `Is` symbols. In the controlled experiment above, neither resolution nor truth-table model checking answered even the single query $Is_{1,1,1}$ within ten seconds. This is expected: resolution repeatedly generates and compares resolvents from a clause set that already has more than ten thousand clauses, while truth-table entailment may enumerate assignments over hundreds of symbols. I therefore interrupted both methods at the stated limit rather than attempting a full-grid solve.

**(b) Forward chaining on the full grid --** Implement `solve_full_grid_fc()` in sudoku_solver.py. Using your above `build_definite_kb` and the library's `pl_fc_entails` (no need to reimplement them), solve the whole puzzle. Find the value that's entailed for every cell. Reconstruct and display the solved grid.

**(c) Backward chaining -- implement it yourself.**
```python
pl_bc_entails(kb, query) -> bool
```
Implement `pl_bc_entails()` in sudoku_solver.py. Start from the query and recursively try to prove each premise of a rule whose conclusion matches the current goal, bottoming out at known facts. Your function must agree with `pl_fc_entails` on every cell/value pair in this puzzle including correctly returning `False` for values that are *not* part of the solution.

**(d) Backward chaining on the full grid --** Implement `solve_full_grid_bc` in sudoku_solver.py. Using your above `build_definite_kb` and your own `pl_bc_entails`, solve the whole puzzle the same way `solve_full_grid_fc` does: for every cell, try each candidate value until `pl_bc_entails` confirms one. Time both `solve_full_grid_fc` and `solve_full_grid_bc` on the same puzzle and compare. Use your measured result where relevant in Conceptual Question 5.


In [6]:
# Implement solve_full_grid_fc(), pl_bc_entails(), and solve_full_grid_bc()
# in sudoku_solver.py. Rerun the import cell near the top after making changes.


### Verifying the algorithms

Uncomment the following block of code to validate your code.

In [7]:
# Verify solve_full_grid_fc against the puzzle's known solution. Then check
# pl_bc_entails directly, for completeness (it finds the correct value) and
# soundness (it never wrongly confirms an incorrect one), before timing
# solve_full_grid_bc against the same solution.

import time

t0 = time.time()
solved = solve_full_grid_fc(n, box_h, box_w, givens)
fc_time = time.time() - t0
assert solved == puzzle['solution']

definite_kb = build_definite_kb(n, box_h, box_w, givens)

# Completeness: pl_bc_entails must find the correct value for every cell.
for (r, c), v in puzzle['solution'].items():
    assert pl_bc_entails(definite_kb, atom('Is', r, c, v)) == True

# Soundness: pl_bc_entails must not also confirm any incorrect value.
for (r, c), v in puzzle['solution'].items():
    for other_v in range(1, n + 1):
        if other_v != v:
            assert pl_bc_entails(definite_kb, atom('Is', r, c, other_v)) == False

t0 = time.time()
solved_bc = solve_full_grid_bc(n, box_h, box_w, givens)
bc_time = time.time() - t0
assert solved_bc == puzzle['solution']

print(f"solve_full_grid_fc: {fc_time:.2f}s")
print(f"solve_full_grid_bc: {bc_time:.2f}s")


solve_full_grid_fc: 11.38s
solve_full_grid_bc: 1.43s


## Part B: Conceptual Questions

Answer all five questions directly in this notebook. Replace each **Your answer:** placeholder with your own response.


### 1. Detailed Representation Strategy: General vs. Definite (Horn) Encoding

Explain in detail how you formalized the Sudoku puzzle constraints into propositional logic across both Knowledge Base representations:

**(a) General KB Strategy (`build_general_kb`):** Detail how standard Sudoku rules (e.g., at-least-one value per cell, at-most-one value per cell, row/column/box uniqueness) are directly translated into Conjunctive Normal Form (CNF) clauses without structural restrictions.

**(b) Definite KB Strategy (`build_definite_kb`):** Definite/Horn clauses strictly permit at most one positive literal per clause, prohibiting disjunctive constraints like $(Is_{r,c,1} \lor Is_{r,c,2} \lor Is_{r,c,3} \lor ... \lor Is_{r,c,n})$. Explain step-by-step how your encoding deals with this issue.


**Answer:**

**(a) General KB.** I use only the $Is_{r,c,v}$ family because ordinary negation is available in unrestricted propositional clauses. For every cell $(r,c)$ I add the at-least-one clause
$$Is_{r,c,1}\lor\cdots\lor Is_{r,c,n}.$$
For every unordered pair $v<w$ I add
$$\neg Is_{r,c,v}\lor\neg Is_{r,c,w},$$
which is the pairwise at-most-one encoding. For every pair of peer cells $a,b$ (same row, column, or box) and every value $v$, I add
$$\neg Is_{a,v}\lor\neg Is_{b,v}.$$
The implementation constructs a set of unique peers first, so a pair that shares both a row and a box is not duplicated. Finally, each given $(r,c)=v$ is added as the unit clause $Is_{r,c,v}$. `PropKB.tell` stores these as CNF clauses.

**(b) Definite KB.** Here `Not` is a second family of *positive propositional symbols*; it is not the syntactic negation operator. Each given is the fact $Is_{r,c,v}$. I then add three kinds of definite rules:

1. $Is_{r,c,v}\Rightarrow Not_{r,c,w}$ for every $w\ne v$ (one cell cannot have two values).
2. $Is_{r,c,v}\Rightarrow Not_{r',c',v}$ for every row/column/box peer $(r',c')$ (peer uniqueness).
3. For each candidate $v$,
$$\bigwedge_{w\ne v} Not_{r,c,w}\Rightarrow Is_{r,c,v}.$$

The third rule replaces the forbidden at-least-one disjunction operationally: once eight candidates have been eliminated, the ninth is inferred. Every rule has a conjunction of positive atoms as its premise and exactly one positive atom as its conclusion, so `PropDefiniteKB.tell` accepts it. Repeated propagation of the elimination and last-candidate rules solves the supplied puzzles without reading their `solution` fields.

### 2. Theoretical Completeness vs. Computational Tractability

Model checking and resolution-refutation are sound and complete—they are guaranteed to terminate with a correct answer for any propositional KB. Despite this guarantee, explain whether you would use either as the default algorithm for solving Sudoku puzzles. *(Hint: Consider space/time complexity and state-space growth, and use your observations from the experiment above where relevant.)*


**Answer:**

I would not use either method as the default Sudoku solver, even though both are sound and complete. The general KB for the selected puzzle has 10,317 clauses and 729 `Is` symbols. Truth-table entailment is exponential in the number of symbols: in the worst case it considers $2^{729}$ assignments for one query, and a full grid needs many queries. Resolution avoids explicit models but repeatedly resolves compatible clause pairs; the number and size of generated resolvents can grow combinatorially, while duplicate/subsumption checks also become expensive.

The controlled run above reflects those bounds: neither method returned one cell query within ten seconds. Soundness and completeness describe correctness, not useful runtime. For these puzzles I would use Horn inference, whose rule-firing closure is polynomial and exploits the elimination structure directly (or a CSP/SAT solver for harder Sudoku instances).

### 3. Backward Chaining: Design, Pseudocode, and Challenges

Write pseudocode for `pl_bc_entails(kb, query)`, the backward-chaining algorithm you implemented. Show, at a level of detail that reveals the algorithm's structure (not full Python), how the function checks whether the query is already a known fact, finds candidate rules whose conclusion matches the current goal, recursively proves each premise of such a rule, and combines results—both across the premises of one rule and across multiple candidate rules—to reach a single boolean answer.

Then, in your own words, discuss the design challenges you had to work through to make your algorithm both correct and guaranteed to terminate on every puzzle, and explain how your pseudocode addresses them.


**Answer:**

The implementation uses *tabled backward chaining*. Its backward traversal is written with an explicit stack rather than Python recursion so the large cyclic dependency graph cannot exceed Python's recursion limit; it is equivalent to the following recursive pseudocode:

```text
BC-ENTAILS(KB, query):
    facts <- all fact clauses in KB
    rules_by_conclusion <- index every (premises => conclusion)
    relevant_goals <- empty set
    relevant_rules <- empty list

    EXPAND(goal):
        if goal is already in relevant_goals: return
        add goal to relevant_goals
        if goal is a known fact: return true
        for each rule (p1 AND ... AND pk => goal):     # alternatives are OR
            add rule to relevant_rules
            for each premise pi:                       # premises are AND
                EXPAND(pi)

    EXPAND(query)

    known <- relevant facts
    repeat until no change:
        for each relevant rule (p1 AND ... AND pk => q):
            if every pi is in known:
                add q to known
                remember this rule as q's proof
    return (query is in known)
```

The main challenge is cyclic support. The rules naturally create paths such as `Is -> Not -> Is`; a naive recursive function can loop forever, and simply returning `False` and caching it when a goal is already on the call stack can make the answer depend on rule order. A goal that is temporarily blocked by a cycle may later become provable through another rule. The visited table terminates dependency expansion, and the least-fixpoint evaluation only derives a conclusion after all premises are already established, so a cycle alone never counts as proof. The stored rule records also form an acyclic proof-by-derivation-order for tutor mode. Across multiple queries on one KB, successful results are safely reused because Horn inference is monotonic.

### 4. Expressive Limits of Horn Logic

Named elimination techniques such as Naked Pairs and X-Wing can, in fact, be encoded as definite clauses, using the same `Is`/`Not` vocabulary as your `build_definite_kb`. Work out how you would encode one of these techniques as definite clauses, and discuss the consequences of doing so.


**Answer:**

Consider a Naked Pair in one row (the same construction works for a column or box). Let cells $a$ and $b$ have only candidates $x$ and $y$. With the assignment's vocabulary, define the positive Horn premise
$$P=\left(\bigwedge_{z\notin\{x,y\}}Not_{a,z}\right)\land
     \left(\bigwedge_{z\notin\{x,y\}}Not_{b,z}\right).$$
For every other cell $d$ in that row, add two definite clauses:
$$P\Rightarrow Not_{d,x},\qquad P\Rightarrow Not_{d,y}.$$
Although the conclusions begin with the word `Not`, each is a single positive atom in this representation, so the clauses are definite. In a consistent Sudoku, the two cells must consume $x$ and $y$ in some order; therefore neither value can occur elsewhere in the unit.

The cost is combinatorial rule growth. We must instantiate rules for every unit, pair of cells, pair of values, and target cell; many rules overlap between rows, columns, and boxes. More named techniques increase memory, indexing work, and the size of reasoning traces. The encoding remains Horn and explainable, but it starts to resemble a hand-built rule engine, so for advanced techniques a compact CSP or SAT encoding is usually easier to maintain.

### 5. Data-Driven vs. Goal-Driven Performance

Forward chaining (data-driven) and backward chaining (goal-driven) are both sound and complete for Horn KBs, but their execution runtimes vary depending on the target query.

**(a)** Describe a scenario—in terms of total KB size versus the query-relevant subset—where backward chaining is significantly faster than forward chaining.

**(b)** Describe a scenario where backward chaining offers no performance advantage, or performs worse than forward chaining.

Use your measured forward- vs. backward-chaining result where relevant.


**Answer:**

**(a)** Backward chaining is attractive when the KB is large but the ancestors of one query form a small subgraph. For example, if thousands of rules describe unrelated boards or independent regions, proving one cell may require only a few givens and elimination rules. A goal-directed procedure visits that subset, whereas forward chaining fires every reachable rule from every fact.

**(b)** It offers little advantage when the query depends on most of the KB, or when we need answers for the whole grid. Separate naive backward searches then rediscover the same premises many times, while one conventional forward closure can answer all later queries by membership lookup. Cycles and many alternative rules can also make failed backward queries expensive.

In the saved timing run above, the provided per-query forward-chaining routine took 11.38 seconds, while my tabled backward chainer took 1.43 seconds for the same full grid. This is implementation-specific: `solve_full_grid_fc` must call the supplied `pl_fc_entails` for many candidate queries, and that function reconstructs its agenda/count table each time. My backward implementation reuses successful tabled proofs across the grid. If forward chaining computed and cached one closure once, it would likely be the better full-grid strategy, while backward chaining would retain its advantage for isolated queries.

## Part C: Streamlit Integration

Wrap your Sudoku solver and inference logic into an interactive Streamlit application, `sudoku_app.py`. Your application must implement the following:

**1. Puzzle selection & visual board display**
- An interactive selector/dropdown to pick any puzzle from `puzzles.json`.
- A visual rendering of the grid that clearly distinguishes the initial *givens* from empty cells.

**2. Full-grid auto-solver, with algorithm selection**
- A control (e.g. radio buttons) letting the user choose forward chaining (`solve_full_grid_fc`) or backward chaining (`solve_full_grid_bc`) before solving.
- A button that solves the full grid with the chosen algorithm, renders the solved state, and displays how long the solve took -- so the timing gap from task (d) is visible in the app, not just in the notebook.

**3. Targeted cell entailment query**
- Inputs for row ($r$), column ($c$), and value ($v$).
- A button that checks whether $\mathit{Is}_{rcv}$ is entailed (using `pl_bc_entails`) and displays the boolean verdict (`True` / `False`).

**4. Reasoning trace ("tutor mode")**
- Instrument your chosen inference algorithm (forward or backward chaining) to record the reasoning steps it takes while answering a query.
- Present that trace in a human-readable format -- not a raw Python string or internal symbol dictionary. For example: expandable cards/accordions showing the rule-firing sequence (*"Inferred $\mathit{Not}_{1,2,3}$ because row 1 already contains value 3"* $\implies$ *"Deduce $\mathit{Is}_{1,2,4}$ as the last remaining candidate"*), or plain-English sentences explaining each elimination by row, column, or box constraint.

Import `atom`, `build_definite_kb`, `build_general_kb`, `solve_full_grid_fc`, `solve_full_grid_bc`, and `pl_bc_entails` from `sudoku_solver.py`. Do not copy or rewrite these core functions in the Streamlit file. You may add app-specific helper functions where needed for the interface or reasoning trace.

Refer to the provided `StreamlitDeploymentGuide.pdf` guide to deploy your code and include the link for your Streamlit app below.


## Submission

Submit only the following three files:

1. `Sudoku_Assignment.ipynb` — with all required cells run, outputs visible, and all conceptual questions answered.
2. `sudoku_solver.py` — containing your implementation of the knowledge-base and inference functions.
3. `sudoku_app.py` — containing your Streamlit application.

The provided support files are required to run the assignment but are not part of the student submission.

### Deployed Streamlit app URL

Paste your deployed Streamlit app URL here:

`https://...`
